In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, roc_auc_score


In [2]:
#garantia de reprodutibilidade
#uso do 42 pois é mais comum em pesquisas
#quero garantir que os resultados sejam auditáveis
#quero uma prova científica sólida toda vez que rodar o modelo, garantindo eficácia
torch.manual_seed(42) #semente da cpu e gpu
np.random.seed(42) #semente para numpy


In [3]:
torch.cuda.manual_seed_all(42) #semente para gpu, focado no hardware para desempenho
torch.backends.cudnn.deterministic = True #forço algoritmos determinísticos. Desativa otimização da NVIDIA


In [4]:
X_train_np = np.load("../artifacts/data/X_train.npy")
X_test_np  = np.load("../artifacts/data/X_test.npy")
y_train_np = np.load("../artifacts/data/y_train.npy")
y_test_np  = np.load("../artifacts/data/y_test.npy")


In [5]:
X_train_t = torch.tensor(X_train_np, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_np, dtype=torch.float32)

y_train_t = torch.tensor(y_train_np, dtype=torch.float32)
y_test_t  = torch.tensor(y_test_np, dtype=torch.float32)


In [6]:
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=256,
    shuffle=True
)


In [7]:
#adição do dropout para maior resiliência da rede 
#quero evitar overfitting (modelo decorando dados)
#não quero que a rede dependa demais de uma única característica (como apenas números de pacotes)
#forço a rede a aprender padrões distribuídos e mais complexos, pois com um dataset como o UNSW, é fácil do modelo decorar ao invés de aprender
#com o dropout, o modelo é capaz de detectar variações de ataques que ele ainda não viu

class RobustMLP(nn.Module):
    def __init__(self, input_dim): #entram os dados brutos (194 características)
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(), #extraí padrões básicos
            nn.Dropout(0.3), #bagunço o aprendizado para forçar robustez

            nn.Linear(128, 64),
            nn.ReLU(), #refino padrões 
            nn.Dropout(0.3),#mais um desafio aos neurônios

            nn.Linear(64, 1) #veredito
        )

    def forward(self, x):
        return self.net(x).squeeze()

#quero um modelo
#1)Atento: não ignora ataque (graças ao peso das classes)
#2)Inteligente: não decora dados inúteis (graças ao dropout)
#3)Eficiente: equilíbrio entre recall e fpr

In [8]:
#uno robustez e aprendizado sensível a custo

device = "cuda" if torch.cuda.is_available() else "cpu"

input_size = 194

model = RobustMLP(input_size).to(device)

positivos = np.sum(y_train_np == 1)
negativos = np.sum(y_train_np == 0)

pos_weight = torch.tensor([negativos / positivos]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4 #aplico um castigo ao modelo se os pesos dos neurônios ficarem muito grandes
)                     #mantenho o modelo sob controle, forçando a rede a ser mais simples e eficiente


In [9]:
#aqui tento ao máximo evitar overfitting

epochs = 20
patience = 3 #modelo tem três chances de não melhorar

best_loss = float("inf") #'recorde' do menor erro alcançado
wait = 0 #contador de frustração

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader) #média do erro

    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping acionado") #se o modelo parar de aprender coisas novas, quero que interrompa o processo
            break


Epoch 1/20 - Loss: 0.1204
Epoch 2/20 - Loss: 0.0909
Epoch 3/20 - Loss: 0.0877
Epoch 4/20 - Loss: 0.0866
Epoch 5/20 - Loss: 0.0852
Epoch 6/20 - Loss: 0.0844
Epoch 7/20 - Loss: 0.0844
Epoch 8/20 - Loss: 0.0837
Epoch 9/20 - Loss: 0.0835
Epoch 10/20 - Loss: 0.0829
Epoch 11/20 - Loss: 0.0824
Epoch 12/20 - Loss: 0.0824
Epoch 13/20 - Loss: 0.0824
Epoch 14/20 - Loss: 0.0819
Epoch 15/20 - Loss: 0.0817
Epoch 16/20 - Loss: 0.0817
Epoch 17/20 - Loss: 0.0819
Epoch 18/20 - Loss: 0.0813
Epoch 19/20 - Loss: 0.0817
Epoch 20/20 - Loss: 0.0813


In [10]:
#comportamento do modelo: extremamente vigilante/conservador
#98% de recall torna o modelo quase impenetrável
#custo que se paga: aumento do FPR em 10%
#motivo: ajuste no threshold
#acurácia: modelo sabe diferenciar muito bem as classes

model.eval()

with torch.no_grad():
    logits = model(X_test_t.to(device))
    probs = torch.sigmoid(logits).cpu().numpy()

threshold = 0.32

y_pred = (probs >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test_np, y_pred).ravel()

recall = tp / (tp + fn)
fpr    = fp / (fp + tn)
auc    = roc_auc_score(y_test_np, probs)

print(f"Recall : {recall:.2%}") #sensibildiade: porcentagem total que o modelo conseguiu pegar
print(f"FPR    : {fpr:.2%}") #alarme falso
print(f"ROC AUC: {auc:.3}") #de 0 a 1, as habilidades do modelo em separar as duas classes


Recall : 98.04%
FPR    : 28.08%
ROC AUC: 0.978


In [11]:
#ponto de salvamento da primeira camada 
torch.save(model.state_dict(), "../artifacts/models/detector_stage1.pth")
